Gemma 4 E2B With Hugging Face Transformers

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 用 HuggingFace 官方实现加载 Gemma 4 E2B 作为「参考答案」，用来和本仓库自实现的 Gemma4 逐层/逐输出对拍
model_id = "google/gemma-4-E2B"  # HF Hub 上的模型仓库 ID（需已获授权/可访问）
prompt = "Give me a short introduction to large language models."

tokenizer = AutoTokenizer.from_pretrained(model_id)  # 加载官方分词器
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",   # 自动选择权重精度（如 bf16），省显存
    device_map="auto",    # 自动把各层分配到可用设备(GPU/CPU)
)
model.eval();  # 切到评估模式（关闭 dropout 等）；行尾分号抑制 Jupyter 打印

In [ ]:
device = next(model.parameters()).device  # 取模型第一个参数所在设备，作为输入张量要搬去的目标设备
device

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to(device)  # 编码为张量并搬到模型设备
input_len = inputs["input_ids"].shape[-1]  # 记录 prompt 的 token 长度，便于稍后只截取「新生成」部分

with torch.inference_mode():  # 推理模式，禁用梯度、更省内存
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,          # 最多生成 200 个新 token
        do_sample=False,             # 关闭采样=贪心解码，保证结果确定、可复现（便于对拍）
        pad_token_id=tokenizer.eos_token_id,  # 显式指定 pad token，避免警告
    )

# 只解码新生成的部分（跳过 prompt），并去掉特殊 token
response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
print(response.strip())